In [3]:
import numpy as np
import rebound
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from astropy.timeseries import LombScargle
from scipy.stats import pearsonr

#-----------------------------------------------------------------------------
# Script Variables
#-----------------------------------------------------------------------------
NUM_SIMULATIONS = 1000
MAX_TRANSITS = 100
USE_RESONANCES = True

#-----------------------------------------------------------------------------
# Utility Functions
#-----------------------------------------------------------------------------

def detect_transits(sim, integration_time, dt, max_transits, planet_index):
    transit_times = []
    star = sim.particles[0]
    planet = sim.particles[planet_index]
    prev_y = planet.y - star.y
    
    time = 0
    while time < integration_time and len(transit_times) < max_transits:
        sim.integrate(sim.t + dt)
        time = sim.t
        curr_y = planet.y - star.y
        if prev_y * curr_y < 0 and (planet.x - star.x) > 0:
            weight = -prev_y / (curr_y - prev_y)
            precise_time = (time - dt) + (weight * dt)
            transit_times.append(precise_time)
        prev_y = curr_y
    return np.array(transit_times)

def detrend_ttv(ttv_data):
    N = len(ttv_data)
    transit_numbers = np.arange(N)
    P = np.polyfit(transit_numbers, ttv_data, 1)
    return ttv_data - (P[0] * transit_numbers + P[1])

#-----------------------------------------------------------------------------
# PHASE 1: GENERATE DATA
#-----------------------------------------------------------------------------

def generate_simulation_data(num_simulations, max_transits):
    print(f"Generating {num_simulations} simulations...")
    all_features, all_masses, all_ttvs = [], [], []
    resonances = [1.5, 2.0, 3.0]
    
    for i in range(num_simulations):
        if (i + 1) % 50 == 0: print(f"  Simulation {i+1}...")
        try:
            m_b, P_b, e_b = np.random.uniform(1e-5, 1e-4), np.random.uniform(2*np.pi, 4*np.pi), np.random.uniform(0.0, 0.02)
            
            if USE_RESONANCES:
                ratio = np.random.choice(resonances) + np.random.uniform(-0.07, 0.07)
            else:
                ratio = np.random.uniform(1.25, 2.5)
            
            m_c, P_c, e_c = np.random.uniform(1e-4, 5e-3), P_b * ratio, np.random.uniform(0.0, 0.05)
            dt = P_b / 50

            # Unperturbed Simulation
            sim_u = rebound.Simulation()    # create simulation                             
            sim_u.integrator = "whfast";    # specify the integrator                        
            sim_u.dt = dt                   # set time step of integrator                   
            sim_u.add(m=1.0);               # add 1 solar mass star to sim                  
            sim_u.add(m=m_b, P=P_b, e=e_b)  # add planet B to sim                           
            sim_u.move_to_com()             # set centre of mass of system as the origin
            transits_u = detect_transits(   # in unperturbed sim, find transits of planet B
                sim_u,                  # simulation to use
                P_b*(max_transits+5),   # integration time 
                dt,                     # integrator time step 
                max_transits,           # max number of transits we're looking for  
                1                       # index of transiting planet in simulation  
            )   

            # Perturbed Simulation  
            sim_p = rebound.Simulation()    # create simulation                            
            sim_p.integrator = "whfast";    # specify the integrator                       
            sim_p.dt = dt                   # set time step of integrator                  
            sim_p.add(m=1.0);               # add 1 solar mass star to sim                 
            sim_p.add(m=m_b, P=P_b, e=e_b)  # add planet B to sim                          
            sim_p.add(m=m_c, P=P_c, e=e_c)  # in unperturbed sim, find transits of planet B
            sim_p.move_to_com()             # set centre of mass of system as the origin
            transits_p = detect_transits(
                sim_p,                  # simulation to use
                P_b*(max_transits+5),   # integration time
                dt,                     # integrator time step
                max_transits,           # max number of transits we're looking for  
                1                       # index of transiting planet in simulation
            )
            if len(transits_u) >= max_transits and len(transits_p) >= max_transits:
                ttv_vec = detrend_ttv(transits_p[:max_transits] - transits_u[:max_transits])
                
                freqs = np.linspace(1/50, 1/3, 40)
                power = LombScargle(np.arange(max_transits), ttv_vec).power(freqs)
                
                
                features = np.hstack(([np.std(ttv_vec)], power, [P_b, ratio, e_b]))
                all_features.append(features)
                all_masses.append(m_c)
                all_ttvs.append(ttv_vec)
        except Exception: continue

    return np.array(all_features), np.array(all_masses), np.array(all_ttvs)

#-----------------------------------------------------------------------------
# PHASE 2: MODEL
#-----------------------------------------------------------------------------

class MassPredictor(nn.Module):
    def __init__(self, input_size):
        super(MassPredictor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.1), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.1),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.1),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.network(x)

def train_model(model, X_train, y_train, X_val, y_val, epochs=200):
    X_train_t, y_train_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_val_t, y_val_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
    
    criterion = nn.HuberLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)
    best_val_loss, best_state = float('inf'), None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        indices = torch.randperm(len(X_train_t)) # PREVENTS LEARNING ANY KIND OF ORDER
        for i in range(0, len(X_train_t), batch_size):
            batch_indices = indices[i:i+batch_size]
            batch_X, batch_y = X_train_t[batch_indices], y_train_t[batch_indices]
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            optimizer.zero_grad()
            loss.backward() # Backpropagation uses calculus chain rule
            optimizer.step() # Weights update
            train_loss += loss.item()
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t)
        
        scheduler.step(val_loss)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Val Loss: {val_loss.item():.6f}')
        
        if val_loss < best_val_loss:
            best_val_loss, patience_counter = val_loss, 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= max_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    if best_model_state:
        model.load_state_dict(best_model_state)
    return model

#-----------------------------------------------------------------------------
# PHASE 3: EXECUTION & ERROR ANALYSIS
#-----------------------------------------------------------------------------

if __name__ == "__main__":
    X_raw, y_raw, ttvs = generate_simulation_data(NUM_SIMULATIONS, MAX_TRANSITS)
    y_log = np.log10(y_raw)
    
    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X_raw)
    
    # Split indices to keep track of raw data for analysis
    indices = np.arange(len(X_scaled))
    idx_train_val, idx_test = train_test_split(indices, test_size=0.2, random_state=42)
    idx_train, idx_val = train_test_split(idx_train_val, test_size=0.2, random_state=42)

    model = MassPredictor(X_scaled.shape[1])
    model = train_model(model, X_scaled[idx_train], y_log[idx_train], X_scaled[idx_val], y_log[idx_val])

    # Predictions
    model.eval()
    with torch.no_grad():
        test_preds = model(torch.tensor(X_scaled[idx_test], dtype=torch.float32)).numpy().flatten()
    
    y_test_actual = y_log[idx_test]
    residuals = test_preds - y_test_actual

    # --- SAVE SECTION ---
    torch.save(model.state_dict(), "ttv_mass_model.pth")
    joblib.dump(scaler_X, "scaler_X.pkl")
    print("\nModel and Scaler saved.")

    # --- ERROR ANALYSIS PLOTS ---
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Binned Median Error (from old code suggestion)
    nbins = 8
    bins = np.linspace(y_test_actual.min(), y_test_actual.max(), nbins+1)
    bin_centers = 0.5*(bins[:-1] + bins[1:])
    med, mad = [], []
    for i in range(nbins):
        sel = (y_test_actual >= bins[i]) & (y_test_actual < bins[i+1])
        if sel.sum() > 0:
            m = np.median(residuals[sel])
            med.append(m)
            mad.append(np.median(np.abs(residuals[sel] - m)))
        else: med.append(np.nan); mad.append(np.nan)

    ax[0].errorbar(bin_centers, med, yerr=mad, fmt='o-', color='teal', capsize=5)
    ax[0].axhline(0, color='black', lw=1, ls='--')
    ax[0].set_title("Binned Median Residuals ± MAD")
    ax[0].set_xlabel("True Mass (Log10)"); ax[0].set_ylabel("Error (dex)")

    # 2. Predicted vs Actual
    ax[1].scatter(y_test_actual, test_preds, alpha=0.5)
    ax[1].plot([y_test_actual.min(), y_test_actual.max()], [y_test_actual.min(), y_test_actual.max()], 'r--')
    ax[1].set_title("Predicted vs Actual Mass")
    ax[1].set_xlabel("True Mass"); ax[1].set_ylabel("Predicted Mass")
    plt.show()

    # --- RESONANCE ANALYSIS ---
    test_ratios = X_raw[idx_test, -2] # Extract 'ratio' column
    is_res = np.any([np.abs(test_ratios - r) < 0.05 for r in [1.5, 2.0, 3.0]], axis=0)
    
    print(f"\nOverall MAE: {mean_absolute_error(y_test_actual, test_preds):.4f}")
    if is_res.any():
        print(f"Resonant MAE: {mean_absolute_error(y_test_actual[is_res], test_preds[is_res]):.4f}")
    if (~is_res).any():
        print(f"Non-Resonant MAE: {mean_absolute_error(y_test_actual[~is_res], test_preds[~is_res]):.4f}")

Generating 1000 simulations...
  Simulation 50...
  Simulation 100...
  Simulation 150...
  Simulation 200...
  Simulation 250...
  Simulation 300...
  Simulation 350...
  Simulation 400...
  Simulation 450...
  Simulation 500...
  Simulation 550...
  Simulation 600...
  Simulation 650...
  Simulation 700...
  Simulation 750...
  Simulation 800...
  Simulation 850...
  Simulation 900...
  Simulation 950...
  Simulation 1000...


NameError: name 'batch_size' is not defined